In [2]:
import pandas as pd
import numpy as np
import ipaddress
from pathlib import Path

OUT_DIR = Path("/home/chupchik/Рабочий стол/fisrt_stepD/output")
parquet_path = OUT_DIR / "ait_ads_wazuh.parquet"

df = pd.read_parquet(parquet_path)
print("Loaded:", df.shape)
df.head(3)

Loaded: (2600263, 14)


,file,timestamp,location,agent_id,agent_name,agent_ip,hostname,program,full_log,rule_id,rule_description,rule_groups,rule_level,rule_groups_str
0,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:32 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
1,fox_wazuh.json,2022-01-15 02:32:32+00:00,/var/log/syslog,6,wazuh-client,192.168.128.170,taylorcruz-mail,freshclam,Jan 15 02:32:32 taylorcruz-mail freshclam[2851...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"
2,fox_wazuh.json,2022-01-15 02:32:37+00:00,/var/log/syslog,18,wazuh-client,172.17.131.81,mail,freshclam,Jan 15 02:32:37 mail freshclam[29266]: Sat Jan...,52507,ClamAV database update,"[clamd, freshclam, virus]",3,"clamd,freshclam,virus"


In [4]:
# timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df = df.dropna(subset=["timestamp"]).copy()

# y_priority
def map_priority(level):
    if pd.isna(level):
        return None
    level = float(level)
    if level <= 3: return "low"
    if level <= 6: return "medium"
    if level <= 10: return "high"
    return "critical"

df["y_priority"] = df["rule_level"].apply(map_priority)
df = df.dropna(subset=["y_priority"]).copy()

# agent_ip + internal/external
df["agent_ip"] = df["agent_ip"].astype(str)

def is_internal_ip(ip):
    try:
        return int(ipaddress.ip_address(ip).is_private)
    except:
        return 0

df["is_internal_ip"] = df["agent_ip"].apply(is_internal_ip).astype("int8")

df["y_priority"].value_counts()

y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64

In [5]:
df["hour"] = df["timestamp"].dt.hour.astype("int16")
df["dow"] = df["timestamp"].dt.dayofweek.astype("int16")
df["is_weekend"] = df["dow"].isin([5,6]).astype("int8")
df["is_night"] = (df["hour"] < 6).astype("int8")

In [6]:
df["rule_id"] = pd.to_numeric(df["rule_id"], errors="coerce").fillna(-1).astype("int32")

df = df.sort_values(["agent_ip", "timestamp"]).reset_index(drop=True)

tmp = (
    df.groupby("agent_ip")
      .rolling("5min", on="timestamp")["rule_id"]
      .count()
      .reset_index(level=0, drop=True)
)

df["ip_burst_5m"] = tmp.to_numpy(dtype=np.int16)
df["log_ip_burst_5m"] = np.log1p(df["ip_burst_5m"]).astype("float32")

df[["agent_ip", "timestamp", "ip_burst_5m"]].head(10)

/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,agent_ip,timestamp,ip_burst_5m
0,10.132.56.1,2022-01-19 01:02:04.669876+00:00,1
1,10.132.56.1,2022-01-19 01:02:04.669876+00:00,2
2,10.132.56.1,2022-01-19 01:02:04.676411+00:00,3
3,10.132.56.1,2022-01-19 01:02:04.676411+00:00,4
4,10.132.56.1,2022-01-19 01:02:04.704695+00:00,5
5,10.132.56.1,2022-01-19 01:02:04.704695+00:00,6
6,10.132.56.1,2022-01-19 01:02:04.790702+00:00,7
7,10.132.56.1,2022-01-19 01:02:04.790702+00:00,8
8,10.132.56.1,2022-01-19 01:02:04.805899+00:00,9
9,10.132.56.1,2022-01-19 01:02:04.805899+00:00,10


In [7]:
df["rule_id"] = pd.to_numeric(df["rule_id"], errors="coerce").fillna(-1).astype("int32")

df = df.sort_values(["agent_ip", "timestamp"]).reset_index(drop=True)

tmp = (
    df.groupby("agent_ip")
      .rolling("5min", on="timestamp")["rule_id"]
      .count()
      .reset_index(level=0, drop=True)
)

df["ip_burst_5m"] = tmp.to_numpy(dtype=np.int16)
df["log_ip_burst_5m"] = np.log1p(df["ip_burst_5m"]).astype("float32")

df[["agent_ip", "timestamp", "ip_burst_5m"]].head(10)

/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/home/chupchik/Рабочий стол/fisrt_stepD/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


,agent_ip,timestamp,ip_burst_5m
0,10.132.56.1,2022-01-19 01:02:04.669876+00:00,1
1,10.132.56.1,2022-01-19 01:02:04.669876+00:00,2
2,10.132.56.1,2022-01-19 01:02:04.676411+00:00,3
3,10.132.56.1,2022-01-19 01:02:04.676411+00:00,4
4,10.132.56.1,2022-01-19 01:02:04.704695+00:00,5
5,10.132.56.1,2022-01-19 01:02:04.704695+00:00,6
6,10.132.56.1,2022-01-19 01:02:04.790702+00:00,7
7,10.132.56.1,2022-01-19 01:02:04.790702+00:00,8
8,10.132.56.1,2022-01-19 01:02:04.805899+00:00,9
9,10.132.56.1,2022-01-19 01:02:04.805899+00:00,10


In [8]:
df["rule_groups_str"] = df["rule_groups_str"].fillna("").astype(str)

df["grp_attack"] = df["rule_groups_str"].str.contains("attack", case=False).astype("int8")
df["grp_recon"]  = df["rule_groups_str"].str.contains("recon", case=False).astype("int8")
df["grp_scan"]   = df["rule_groups_str"].str.contains("scan", case=False).astype("int8")
df["grp_auth"]   = df["rule_groups_str"].str.contains("authentication", case=False).astype("int8")
df["grp_ids"]    = df["rule_groups_str"].str.contains("ids", case=False).astype("int8")

In [9]:
df["location"] = df["location"].fillna("").astype(str)

df["is_apache"]   = df["location"].str.contains("apache", case=False).astype("int8")
df["is_suricata"] = df["location"].str.contains("suricata", case=False).astype("int8")
df["is_syslog"]   = df["location"].str.contains("syslog", case=False).astype("int8")
df["is_mail"]     = df["location"].str.contains("mail", case=False).astype("int8")

In [10]:
df["full_log"] = df["full_log"].fillna("").astype(str)

# HTTP коды
df["has_400"] = df["full_log"].str.contains(r" 400 ").astype("int8")
df["has_401"] = df["full_log"].str.contains(r" 401 ").astype("int8")
df["has_403"] = df["full_log"].str.contains(r" 403 ").astype("int8")
df["has_500"] = df["full_log"].str.contains(r" 500 ").astype("int8")

# методы
df["has_post"] = df["full_log"].str.contains("POST").astype("int8")
df["has_get"]  = df["full_log"].str.contains("GET").astype("int8")

# паттерны
df["has_sql"]   = df["full_log"].str.contains(r"select|union|sql", case=False, regex=True).astype("int8")
df["has_admin"] = df["full_log"].str.contains(r"/admin", case=False).astype("int8")
df["has_wp"]    = df["full_log"].str.contains(r"wp-", case=False).astype("int8")

In [11]:
y = df["y_priority"].astype(str)

cat_cols = ["location", "program", "agent_name", "hostname"]
cat_cols = [c for c in cat_cols if c in df.columns]

num_cols = [
    "hour","dow","is_weekend","is_night",
    "is_internal_ip",
    "log_ip_burst_5m","log_rule_burst_5m",
    "grp_attack","grp_recon","grp_scan","grp_auth","grp_ids",
    "is_apache","is_suricata","is_syslog","is_mail",
    "has_400","has_401","has_403","has_500",
    "has_post","has_get",
    "has_sql","has_admin","has_wp"
]
num_cols = [c for c in num_cols if c in df.columns]

X = df[cat_cols + num_cols].copy()

for c in cat_cols:
    X[c] = X[c].fillna("NA").astype(str)
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)

print("X:", X.shape, "y:", y.shape)
print("y dist:\n", y.value_counts())

X: (2600263, 28) y: (2600263,)
y dist:
 y_priority
medium      1873973
low          592722
high         131901
critical       1667
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train dist:\n", y_train.value_counts(normalize=True))

Train: (2080210, 28) Test: (520053, 28)
Train dist:
 y_priority
medium      0.720686
low         0.227947
high        0.050726
critical    0.000641
Name: proportion, dtype: float64


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

gb = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=400,
    max_depth=None,
    min_samples_leaf=30,
    l2_regularization=1e-6,
    random_state=42
)

model_gb = Pipeline(steps=[
    ("prep", preprocess),
    ("gb", gb)
])

custom_weights = {
    "medium": 1.0,
    "low": 1.2,
    "high": 3.0,
    "critical": 30.0   # <-- ослабляй: 8, 6, 4, 3...
}

sw = y_train.map(custom_weights).values

model_gb.fit(X_train, y_train, gb__sample_weight=sw)
print("USING WEIGHTS:", custom_weights)

USING WEIGHTS: {'medium': 1.0, 'low': 1.2, 'high': 3.0, 'critical': 30.0}


In [21]:
type(model_gb.named_steps["gb"])

sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier

In [22]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd

print("USING WEIGHTS:", custom_weights)

pred = model_gb.predict(X_test)  # <-- обязательно заново

print(classification_report(y_test, pred, digits=3))
print("macro-F1:", f1_score(y_test, pred, average="macro"))

labels = ["low", "medium", "high", "critical"]
cm = confusion_matrix(y_test, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

USING WEIGHTS: {'medium': 1.0, 'low': 1.2, 'high': 3.0, 'critical': 30.0}
              precision    recall  f1-score   support

    critical      0.000     0.000     0.000       333
        high      1.000     0.923     0.960     26380
         low      0.998     1.000     0.999    118545
      medium      0.994     0.999     0.997    374795

    accuracy                          0.995    520053
   macro avg      0.748     0.731     0.739    520053
weighted avg      0.994     0.995     0.995    520053

macro-F1: 0.7389052356827741


,pred_low,pred_medium,pred_high,pred_critical
true_low,118545,0,0,0
true_medium,228,374550,0,17
true_high,8,2016,24354,2
true_critical,0,333,0,0


In [23]:
err = X_test.copy()
err["y_true"] = y_test.values

pred = model_gb.predict(X_test)  # <-- заново здесь тоже, для надежности
err["y_pred"] = pred

pred_crit = (err["y_pred"] == "critical").sum()
true_crit = (err["y_true"] == "critical").sum()
tp_crit = ((err["y_true"] == "critical") & (err["y_pred"] == "critical")).sum()

print("USING WEIGHTS:", custom_weights)
print("true_critical:", true_crit)
print("predicted_critical:", pred_crit)
print("TP_critical:", tp_crit)

hc = ((err["y_true"] == "high") & (err["y_pred"] == "critical")).sum()
print("high -> critical errors:", hc)

USING WEIGHTS: {'medium': 1.0, 'low': 1.2, 'high': 3.0, 'critical': 30.0}
true_critical: 333
predicted_critical: 19
TP_critical: 0
high -> critical errors: 2


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
#смотрим какая балансировка будут логичной (я про веса критикал)
classes = np.array(["low","medium","high","critical"])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
balanced_weights = dict(zip(classes, cw))

balanced_weights

{np.str_('low'): np.float64(1.0967476279954531),
 np.str_('medium'): np.float64(0.3468917633529841),
 np.str_('high'): np.float64(4.928426569118943),
 np.str_('critical'): np.float64(389.84445277361317)}